## imports and model loading

In [2]:
from transformer_lens import HookedTransformer
import transformer_lens.utils as utils
import circuitsvis as cv
from transformers import AutoModelForCausalLM, AutoTokenizer
import plotly.express as px
import torch
from functools import partial
from jaxtyping import Float
import tqdm as tqdm
from transformer_lens.hook_points import (
    HookPoint,
) 
import einops
import numpy as np
import json

device = "cpu" # running on my laptop, use cuda if you have a gpu
model_path = "44David/qwen-0.5b-reasoning-v2" # my model uploaded to hf

hf_model = AutoModelForCausalLM.from_pretrained(model_path,).to(device)

tokenizer = AutoTokenizer.from_pretrained(model_path)

model = HookedTransformer.from_pretrained(
    "Qwen/Qwen2.5-0.5B",
    hf_model=hf_model,
    tokenizer=tokenizer,
    device=device,
    center_writing_weights=False, # because we aren't using layernorm
    center_unembed=False,
)

/david/Development/mech-interp-qwen-reasoning-0.5b/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loaded pretrained model Qwen/Qwen2.5-0.5B into HookedTransformer


## logit lens analysis

Logit Lens analysis essentially measures a score of how much the model *wants* to output that specific token at a layer. 
In this case, we look for the think tag token, simply, '<tr'. 
The divergence score we calculate  tells us at each layer how likely the model is to output think tags and therefore reason between the two prompts.

Here, a negative score means the model was more likely to reason on factual tasks, and positive means using think tags on reasoning tasks

In [3]:
def find_decision_layer(model, examples):    
    results = []
    
    for prompt in examples:
        logits, cache = model.run_with_cache(prompt)
        
        decision_pos = len(prompt.split()) - 1  
        
        think_token_id = model.tokenizer.encode('<t')[0] # "<t" since we're looking for <think> tags
        
        layer_logits = []
        for layer in range(model.cfg.n_layers):
            resid = cache["resid_post", layer][0, decision_pos]
            
            layer_logit = model.unembed(model.ln_final(resid))
            think_score = layer_logit[think_token_id].item()
            layer_logits.append(think_score)
            
        results.append(layer_logits)
    
    return results

In [6]:
# use function on 2 example prompts

# open dataset
with open('../interp-dataset.jsonl', 'r') as f:
        data = json.load(f)

reasoning_task = [data['reasoning'][0]['prompt']]
factual_task = [data['factual'][0]['prompt']]

reasoning_scores = (find_decision_layer(model, reasoning_task))
factual_scores = (find_decision_layer(model, factual_task))

mean_reason_score = np.mean(reasoning_scores, axis=0)
mean_fact_score = np.mean(factual_scores, axis=0) 
 
divergence = mean_reason_score - mean_fact_score


print("layer divergence:\n")
for i, div in enumerate(divergence):
    print(f"layer {i:2d}: {div}")


layer divergence:

layer  0: -6.20079779624939
layer  1: -5.267170667648315
layer  2: -2.3411033153533936
layer  3: -1.375074565410614
layer  4: -3.320032447576523
layer  5: -2.239061027765274
layer  6: 1.3569442629814148
layer  7: 1.4395968914031982
layer  8: 0.9058853387832642
layer  9: 0.3655567169189453
layer 10: 0.45757055282592773
layer 11: 2.1146583557128906
layer 12: 0.3274216651916504
layer 13: 1.7283734679222107
layer 14: 1.31028413772583
layer 15: 2.8466164469718933
layer 16: 7.0378782749176025
layer 17: 9.471742391586304
layer 18: 8.067124128341675
layer 19: 9.04071593284607
layer 20: 4.119766354560852
layer 21: 2.9394184350967407
layer 22: 0.16646742820739746
layer 23: 1.9631412029266357


This is pretty interesting, it seems that the model wants to reason on factual questions in the early layers, but then completely flips in the later layers.
The target for the next step (activation patching) will be the layers with the largest divergence values, in descending order: (L17, L19, L18, L16) 

## Activation Patching

For a in depth defintion, look [here](https://dynalist.io/d/n2ZWtnoYHrU1s4vnFSAQ519J#z=qeWBvs-R-taFfcCq-S_hgMqx) 


In [11]:
clean_prompt = "Mary has 5 cases of tennis balls, each case contains 6 balls, meaning she has 30 tennis balls, if she gets one more crate she will have " 
corrupted_prompt = "Mary has 5 cases of tennis balls, each case contains 6 balls, meaning she has 11 tennis balls, if she gets one more crate she will have " 

clean_tokens = model.to_tokens(clean_prompt)
corrupted_tokens = model.to_tokens(corrupted_prompt)


def logits_to_logit_diff(logits, correct_answer="37", incorrect_answer="17"):
    correct_index = model.to_tokens(correct_answer)
    incorrect_index = model.to_tokens(incorrect_answer)
    return logits[0, -1, correct_index] - logits[0, -1, incorrect_index]

clean_logits, clean_cache = model.run_with_cache(clean_tokens)
clean_logit_diff = logits_to_logit_diff(clean_logits)
print(f"clean logit diff: {clean_logit_diff}")


corrupted_logits, corrupted_cache = model.run_with_cache(corrupted_tokens)
corrupted_logit_diff = logits_to_logit_diff(corrupted_logits)
print(f"corrupted logit diff: {corrupted_logit_diff}")


clean logit diff: tensor([[-0.7405,  0.0000]], grad_fn=<SubBackward0>)
corrupted logit diff: tensor([[0.4609, 0.0000]], grad_fn=<SubBackward0>)
